<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/Ceng467_TakeHome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install datasets transformers torch scikit-learn matplotlib seaborn -q

import random
import numpy as np
import torch

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup complete!")
print("GPU available:", torch.cuda.is_available())
print("Seed:", SEED)

Setup complete!
GPU available: True
Seed: 42


In [2]:
# Load IMDb dataset
from datasets import load_dataset
import numpy as np

dataset = load_dataset("imdb")

print(dataset)
print("\n--- Sample training record ---")
print("Text:", dataset["train"][0]["text"][:300], "...")
print("Label:", dataset["train"][0]["label"], "(0=Negative, 1=Positive)")
print("\nTotal training examples:", len(dataset["train"]))
print("Total test examples:", len(dataset["test"]))

# Dataset properties for report
print("\n--- Dataset Properties ---")
print("Task: Binary Sentiment Classification")
print("Classes: Negative (0), Positive (1)")
print("Train set: 25,000 reviews (balanced: 12,500 per class)")
print("Test set:  25,000 reviews (balanced: 12,500 per class)")
print("Unsupervised set: 50,000 reviews (unlabeled, not used)")
print("Avg review length (chars):", int(np.mean([len(x["text"]) for x in dataset["train"]])))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

--- Sample training record ---
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...
Label: 0 (0=Negative, 1=Positive)

Total training examples: 25000
Total test examples: 25000

--- Dataset Properties ---
Task: Binary Sentiment Classification
Classes: Negative (0), Positive (1)
Train set: 25,000 reviews (balanced: 12,500 per class)
Test set:  25,000 reviews (balanced: 12,500 per class)
Unsupervised set: 50,000 reviews (unlabeled, not used)
Avg rev

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Convert to pandas dataframe
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Carve out a validation set from training data (80/20 split)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df["label"]
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

# Check label distribution
print("\nTrain label distribution:")
print(train_df["label"].value_counts())

Train size: 20000
Validation size: 5000
Test size: 25000

Train label distribution:
label
1    10000
0    10000
Name: count, dtype: int64


In [4]:
import re
import string
import nltk
nltk.download('stopwords', quiet=True)

def preprocess_text(text, remove_stopwords=False):
    # Lowercase
    text = text.lower()
    # Remove HTML tags (IMDb reviews often contain <br /> tags)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    if remove_stopwords:
        from nltk.corpus import stopwords
        stop_words = set(stopwords.words('english'))
        text = ' '.join([w for w in text.split() if w not in stop_words])

    return text

# Version 1: with stopwords kept
train_df["text_clean"] = train_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))
val_df["text_clean"]   = val_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))
test_df["text_clean"]  = test_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=False))

# Version 2: stopwords removed (used for preprocessing comparison)
train_df["text_no_sw"] = train_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=True))
val_df["text_no_sw"]   = val_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=True))
test_df["text_no_sw"]  = test_df["text"].apply(lambda x: preprocess_text(x, remove_stopwords=True))

print("Preprocessing done!")
print("\nOriginal sample:")
print(train_df["text"].iloc[0][:200])
print("\nCleaned (with stopwords):")
print(train_df["text_clean"].iloc[0][:200])
print("\nCleaned (without stopwords):")
print(train_df["text_no_sw"].iloc[0][:200])

# Token count comparison
avg_with_sw    = train_df["text_clean"].apply(lambda x: len(x.split())).mean()
avg_without_sw = train_df["text_no_sw"].apply(lambda x: len(x.split())).mean()
print(f"\nAvg tokens with stopwords:    {avg_with_sw:.1f}")
print(f"Avg tokens without stopwords: {avg_without_sw:.1f}")
print(f"Reduction: {(1 - avg_without_sw/avg_with_sw)*100:.1f}%")

Preprocessing done!

Original sample:
I have always been a huge James Bond fanatic! I have seen almost all of the films except for Die Another Day, and The World Is Not Enough. The graphic's for Everything Or Nothing are breathtaking! The

Cleaned (with stopwords):
i have always been a huge james bond fanatic i have seen almost all of the films except for die another day and the world is not enough the graphics for everything or nothing are breathtaking the voic

Cleaned (without stopwords):
always huge james bond fanatic seen almost films except die another day world enough graphics everything nothing breathtaking voice talents wow love pierce brosnan finally bond video game bond enjoyed

Avg tokens with stopwords:    229.9
Avg tokens without stopwords: 121.3
Reduction: 47.2%


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ============================================================
# TOKENIZATION STRATEGY COMPARISON
# Strategy 1: Unigrams only
# Strategy 2: Unigrams + Bigrams
# ============================================================

def train_and_evaluate_lr(X_train, X_val, y_train, y_val, label=""):
    model = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds, average="macro")
    print(f"{label:<45} Acc: {acc:.4f}  Macro-F1: {f1:.4f}")
    return model, preds, acc, f1

print("=" * 65)
print("TOKENIZATION STRATEGY COMPARISON (with stopwords)")
print("=" * 65)

# Unigrams only
tfidf_uni = TfidfVectorizer(max_features=50000, ngram_range=(1,1), min_df=2, max_df=0.95)
X_train_uni = tfidf_uni.fit_transform(train_df["text_clean"])
X_val_uni   = tfidf_uni.transform(val_df["text_clean"])
train_and_evaluate_lr(X_train_uni, X_val_uni, train_df["label"], val_df["label"],
                      label="Unigrams only")

# Unigrams + Bigrams
tfidf_bi = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, max_df=0.95)
X_train_bi = tfidf_bi.fit_transform(train_df["text_clean"])
X_val_bi   = tfidf_bi.transform(val_df["text_clean"])
train_and_evaluate_lr(X_train_bi, X_val_bi, train_df["label"], val_df["label"],
                      label="Unigrams + Bigrams")

print()
print("=" * 65)
print("PREPROCESSING COMPARISON (Unigrams + Bigrams)")
print("=" * 65)

# With stopwords
_, _, acc_with_sw, f1_with_sw = train_and_evaluate_lr(
    X_train_bi, X_val_bi, train_df["label"], val_df["label"],
    label="With stopwords")

# Without stopwords
tfidf_no_sw = TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, max_df=0.95)
X_train_no_sw = tfidf_no_sw.fit_transform(train_df["text_no_sw"])
X_val_no_sw   = tfidf_no_sw.transform(val_df["text_no_sw"])
_, _, acc_no_sw, f1_no_sw = train_and_evaluate_lr(
    X_train_no_sw, X_val_no_sw, train_df["label"], val_df["label"],
    label="Without stopwords")

# ============================================================
# FINAL MODEL: Unigrams + Bigrams + With Stopwords (best config)
# ============================================================
print("\n--- Final LR Model (Unigrams+Bigrams, with stopwords) ---")
tfidf = tfidf_bi
X_train_tfidf = X_train_bi
X_val_tfidf   = X_val_bi
X_test_tfidf  = tfidf.transform(test_df["text_clean"])

lr_model      = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_model.fit(X_train_tfidf, train_df["label"])
val_preds_lr  = lr_model.predict(X_val_tfidf)
val_acc_lr    = accuracy_score(val_df["label"], val_preds_lr)
val_f1_lr     = f1_score(val_df["label"], val_preds_lr, average="macro")

print(classification_report(val_df["label"], val_preds_lr, target_names=["Negative", "Positive"]))

TOKENIZATION STRATEGY COMPARISON (with stopwords)
Unigrams only                                 Acc: 0.8886  Macro-F1: 0.8886
Unigrams + Bigrams                            Acc: 0.8964  Macro-F1: 0.8964

PREPROCESSING COMPARISON (Unigrams + Bigrams)
With stopwords                                Acc: 0.8964  Macro-F1: 0.8964
Without stopwords                             Acc: 0.8970  Macro-F1: 0.8970

--- Final LR Model (Unigrams+Bigrams, with stopwords) ---
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.90      2500
    Positive       0.89      0.90      0.90      2500

    accuracy                           0.90      5000
   macro avg       0.90      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000



In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ============================================================
# TOKENIZATION STRATEGY NOTE FOR BILSTM:
# Unlike DistilBERT (WordPiece subword tokenization),
# BiLSTM uses simple whitespace tokenization.
# This is the third tokenization strategy in our comparison:
#   1. TF-IDF unigrams (word-level, sparse)
#   2. TF-IDF unigrams+bigrams (word-level + phrase, sparse)
#   3. Whitespace tokenization → integer IDs (BiLSTM)
#   4. WordPiece subword tokenization (DistilBERT)
# ============================================================

def build_vocab(texts, max_vocab=30000):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df["text_clean"])
print(f"Vocabulary size: {len(vocab)}")

# Show tokenization example
sample = train_df["text_clean"].iloc[0]
tokens = sample.split()[:10]
ids    = [vocab.get(t, 1) for t in tokens]
print("\n--- Whitespace Tokenization Example ---")
print("Tokens:", tokens)
print("IDs:   ", ids)

# ---- Dataset ----
class IMDbDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=256):
        self.texts  = texts
        self.labels = labels
        self.vocab  = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = text.split()[:self.max_len]
        ids = [self.vocab.get(t, 1) for t in tokens]
        ids += [0] * (self.max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.encode(self.texts.iloc[idx]), dtype=torch.long)
        y = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return x, y

train_dataset = IMDbDataset(train_df["text_clean"], train_df["label"], vocab)
val_dataset   = IMDbDataset(val_df["text_clean"],   val_df["label"],   vocab)
test_dataset  = IMDbDataset(test_df["text_clean"],  test_df["label"],  vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

print("\nDatasets ready!")

Vocabulary size: 30000

--- Whitespace Tokenization Example ---
Tokens: ['i', 'have', 'always', 'been', 'a', 'huge', 'james', 'bond', 'fanatic', 'i']
IDs:    [10, 25, 203, 76, 4, 650, 561, 1534, 7134, 10]

Datasets ready!


In [8]:
# ---- BiLSTM Model ----
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super(BiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)  # *2 because bidirectional

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        # Concatenate last forward and backward hidden states
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

# ---- Training Function ----
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (preds.argmax(1) == y).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = criterion(preds, y)
            total_loss += loss.item()
            correct += (preds.argmax(1) == y).sum().item()
            all_preds.extend(preds.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / len(loader), correct / len(loader.dataset), all_preds, all_labels

# ---- Train BiLSTM ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

bilstm_model = BiLSTM(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5
print("\nTraining BiLSTM...")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(bilstm_model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(bilstm_model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Using device: cuda

Training BiLSTM...
Epoch 1/5 | Train Loss: 0.6917 | Train Acc: 0.5274 | Val Loss: 0.6917 | Val Acc: 0.5006
Epoch 2/5 | Train Loss: 0.6702 | Train Acc: 0.5818 | Val Loss: 0.6335 | Val Acc: 0.6222
Epoch 3/5 | Train Loss: 0.5435 | Train Acc: 0.7295 | Val Loss: 0.4017 | Val Acc: 0.8200
Epoch 4/5 | Train Loss: 0.3774 | Train Acc: 0.8350 | Val Loss: 0.3161 | Val Acc: 0.8656
Epoch 5/5 | Train Loss: 0.2858 | Train Acc: 0.8825 | Val Loss: 0.2951 | Val Acc: 0.8722


In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Evaluate BiLSTM on validation set
_, val_acc_bilstm, val_preds_bilstm, val_labels_bilstm = evaluate(
    bilstm_model, val_loader, criterion, device
)
val_f1_bilstm = f1_score(val_labels_bilstm, val_preds_bilstm, average="macro")

print("--- BiLSTM Results ---")
print(f"Validation Accuracy: {val_acc_bilstm:.4f}")
print(f"Validation Macro-F1: {val_f1_bilstm:.4f}")
print("\nClassification Report:")
print(classification_report(val_labels_bilstm, val_preds_bilstm, target_names=["Negative", "Positive"]))

--- BiLSTM Results ---
Validation Accuracy: 0.8722
Validation Macro-F1: 0.8722

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.86      0.87      2500
    Positive       0.86      0.88      0.87      2500

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000



In [10]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader

# ---- Tokenizer (WordPiece subword tokenization) ----
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Show WordPiece tokenization example (contrast with whitespace tokenization)
sample = train_df["text_clean"].iloc[0]
wordpiece_tokens = tokenizer.tokenize(sample[:100])
print("--- WordPiece Tokenization Example ---")
print("Input:", sample[:100])
print("Tokens:", wordpiece_tokens)
print(f"Token count (WordPiece): {len(tokenizer.tokenize(sample))}")
print(f"Token count (Whitespace): {len(sample.split())}")

# ---- Dataset ----
class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("\nTokenizing datasets...")
bert_train = BertDataset(train_df["text_clean"], train_df["label"], tokenizer)
bert_val   = BertDataset(val_df["text_clean"],   val_df["label"],   tokenizer)
bert_test  = BertDataset(test_df["text_clean"],  test_df["label"],  tokenizer)

bert_train_loader = DataLoader(bert_train, batch_size=32, shuffle=True)
bert_val_loader   = DataLoader(bert_val,   batch_size=32)
bert_test_loader  = DataLoader(bert_test,  batch_size=32)

print("Tokenization done!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- WordPiece Tokenization Example ---
Input: i have always been a huge james bond fanatic i have seen almost all of the films except for die anot
Tokens: ['i', 'have', 'always', 'been', 'a', 'huge', 'james', 'bond', 'fan', '##atic', 'i', 'have', 'seen', 'almost', 'all', 'of', 'the', 'films', 'except', 'for', 'die', 'an', '##ot']
Token count (WordPiece): 455
Token count (Whitespace): 397

Tokenizing datasets...
Tokenization done!


In [11]:
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW

# Load pretrained DistilBERT
print("Loading DistilBERT model...")
bert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
).to(device)

optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5)

# ---- Train & Evaluate Functions for BERT ----
def train_bert_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate_bert(model, loader, device):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / len(loader.dataset), all_preds, all_labels

# ---- Training ----
EPOCHS_BERT = 3
print("\nTraining DistilBERT...")
for epoch in range(EPOCHS_BERT):
    train_loss, train_acc = train_bert_epoch(bert_model, bert_train_loader, optimizer_bert, device)
    val_loss, val_acc, _, _ = evaluate_bert(bert_model, bert_val_loader, device)
    print(f"Epoch {epoch+1}/{EPOCHS_BERT} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Loading DistilBERT model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training DistilBERT...
Epoch 1/3 | Train Loss: 0.3138 | Train Acc: 0.8637 | Val Loss: 0.2359 | Val Acc: 0.9008
Epoch 2/3 | Train Loss: 0.1793 | Train Acc: 0.9351 | Val Loss: 0.2229 | Val Acc: 0.9128
Epoch 3/3 | Train Loss: 0.1027 | Train Acc: 0.9652 | Val Loss: 0.2654 | Val Acc: 0.9102


In [12]:
# Final evaluation of DistilBERT on validation set
_, val_acc_bert, val_preds_bert, val_labels_bert = evaluate_bert(
    bert_model, bert_val_loader, device
)
val_f1_bert = f1_score(val_labels_bert, val_preds_bert, average="macro")

print("--- DistilBERT Results ---")
print(f"Validation Accuracy: {val_acc_bert:.4f}")
print(f"Validation Macro-F1: {val_f1_bert:.4f}")
print("\nClassification Report:")
print(classification_report(val_labels_bert, val_preds_bert, target_names=["Negative", "Positive"]))

# ---- Summary Table ----
print("\n" + "="*55)
print(f"{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("="*55)
print(f"{'TF-IDF + LogReg':<25} {val_acc_lr:>10.4f} {val_f1_lr:>10.4f}")
print(f"{'BiLSTM':<25} {val_acc_bilstm:>10.4f} {val_f1_bilstm:>10.4f}")
print(f"{'DistilBERT':<25} {val_acc_bert:>10.4f} {val_f1_bert:>10.4f}")
print("="*55)


--- DistilBERT Results ---
Validation Accuracy: 0.9102
Validation Macro-F1: 0.9102

Classification Report:
              precision    recall  f1-score   support

    Negative       0.91      0.91      0.91      2500
    Positive       0.91      0.91      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000


Model                       Accuracy   Macro-F1
TF-IDF + LogReg               0.8964     0.8964
BiLSTM                        0.8722     0.8722
DistilBERT                    0.9102     0.9102


In [13]:
# Misclassification analysis for DistilBERT
import pandas as pd

val_df_reset = val_df.reset_index(drop=True)

results_df = pd.DataFrame({
    "text": val_df_reset["text"],
    "true_label": val_labels_bert,
    "pred_label": val_preds_bert
})

# Filter misclassified examples
misclassified = results_df[results_df["true_label"] != results_df["pred_label"]].reset_index(drop=True)

print(f"Total misclassified: {len(misclassified)} / {len(val_df)} ({len(misclassified)/len(val_df)*100:.1f}%)")

# Error type breakdown
fn = misclassified[misclassified["true_label"] == 1]  # Positive predicted as Negative
fp = misclassified[misclassified["true_label"] == 0]  # Negative predicted as Positive
print(f"False Negatives (Positive → Negative): {len(fn)}")
print(f"False Positives (Negative → Positive): {len(fp)}")

label_map = {0: "Negative", 1: "Positive"}

print("\n" + "="*80)
for i in range(5):
    row = misclassified.iloc[i]
    print(f"\n--- Example {i+1} ---")
    print(f"True Label : {label_map[row['true_label']]}")
    print(f"Predicted  : {label_map[row['pred_label']]}")
    print(f"Text       : {row['text'][:400]}...")
    print("="*80)

# ---- Common Error Pattern Analysis ----
print("\n--- Common Error Patterns ---")

# Pattern 1: Irony/Sarcasm — positive review but lots of negative words
import re
neg_words = ["bad", "terrible", "horrible", "awful", "worst", "boring", "stupid", "waste"]

def count_neg_words(text):
    text = text.lower()
    return sum(1 for w in neg_words if w in text)

misclassified["neg_word_count"] = misclassified["text"].apply(count_neg_words)
correct = results_df[results_df["true_label"] == results_df["pred_label"]].reset_index(drop=True)
correct["neg_word_count"] = correct["text"].apply(count_neg_words)

print(f"Avg negative words in misclassified: {misclassified['neg_word_count'].mean():.2f}")
print(f"Avg negative words in correct:       {correct['neg_word_count'].mean():.2f}")

# Pattern 2: Review length
misclassified["length"] = misclassified["text"].apply(len)
correct["length"]       = correct["text"].apply(len)
print(f"\nAvg review length (misclassified): {misclassified['length'].mean():.0f} chars")
print(f"Avg review length (correct):       {correct['length'].mean():.0f} chars")

# Pattern 3: Mixed sentiment — contains both positive and negative words
pos_words = ["great", "excellent", "amazing", "wonderful", "fantastic", "brilliant"]
def count_pos_words(text):
    text = text.lower()
    return sum(1 for w in pos_words if w in text)

misclassified["pos_word_count"] = misclassified["text"].apply(count_pos_words)
mixed = misclassified[
    (misclassified["neg_word_count"] > 0) & (misclassified["pos_word_count"] > 0)
]
print(f"\nMixed sentiment reviews in misclassified: {len(mixed)} / {len(misclassified)} ({len(mixed)/len(misclassified)*100:.1f}%)")

Total misclassified: 449 / 5000 (9.0%)
False Negatives (Positive → Negative): 214
False Positives (Negative → Positive): 235


--- Example 1 ---
True Label : Positive
Predicted  : Negative
Text       : Yowsa! If you REALLY want some ACTION, check out the babes and bombs on this non-stop thriller! Veteran star MARTIN SHEEN leads a trio of supermodels on a mission to stop nuclear terrorism... but director Dean Hamilton doesn't let this heavy plotline get in the way of massive doses of TEENSY-SWIMSUIT scenes, jiggly beach jogs, hubba-hubba hot tubs and the like! Want action? You'll get more of it h...

--- Example 2 ---
True Label : Positive
Predicted  : Negative
Text       : Any movie that shows federal PIGs (Persons In Government) to be the power-mad threats they are in real life has a lot to recommend it to me.<br /><br />Alas, the script supervision and editing and even, at times, the directing are flawed so there will be people who will disparage the whole movie and ignore the good m

In [14]:
# Final evaluation on TEST SET (used only once!)
print("="*55)
print("FINAL TEST SET EVALUATION")
print("="*55)

# TF-IDF + LogReg
test_preds_lr = lr_model.predict(X_test_tfidf)
test_acc_lr = accuracy_score(test_df["label"], test_preds_lr)
test_f1_lr = f1_score(test_df["label"], test_preds_lr, average="macro")

# BiLSTM
_, test_acc_bilstm, test_preds_bilstm, test_labels = evaluate(
    bilstm_model, test_loader, criterion, device
)
test_f1_bilstm = f1_score(test_labels, test_preds_bilstm, average="macro")

# DistilBERT
_, test_acc_bert, test_preds_bert, test_labels_bert = evaluate_bert(
    bert_model, bert_test_loader, device
)
test_f1_bert = f1_score(test_labels_bert, test_preds_bert, average="macro")

print(f"\n{'Model':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print("="*55)
print(f"{'TF-IDF + LogReg':<25} {test_acc_lr:>10.4f} {test_f1_lr:>10.4f}")
print(f"{'BiLSTM':<25} {test_acc_bilstm:>10.4f} {test_f1_bilstm:>10.4f}")
print(f"{'DistilBERT':<25} {test_acc_bert:>10.4f} {test_f1_bert:>10.4f}")
print("="*55)


FINAL TEST SET EVALUATION

Model                       Accuracy   Macro-F1
TF-IDF + LogReg               0.8913     0.8913
BiLSTM                        0.8622     0.8622
DistilBERT                    0.9094     0.9094


In [15]:
# ---- Tokenization Strategy Comparison Summary ----
print("=" * 70)
print("TOKENIZATION STRATEGY COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Strategy':<30} {'Vocab':<15} {'OOV Handling':<20} {'Used In'}")
print("-" * 70)
print(f"{'TF-IDF Unigrams':<30} {'50K sparse':<15} {'Ignored':<20} {'LogReg (baseline)'}")
print(f"{'TF-IDF Unigrams+Bigrams':<30} {'50K sparse':<15} {'Ignored':<20} {'LogReg (final)'}")
print(f"{'Whitespace → integer IDs':<30} {'30K':<15} {'<UNK> token':<20} {'BiLSTM'}")
print(f"{'WordPiece subword':<30} {'30K subword':<15} {'Subword split':<20} {'DistilBERT'}")
print("=" * 70)
print("\nKey observation: WordPiece handles OOV words by splitting into subwords")
print("e.g. 'fanatic' → ['fan', '##atic'], eliminating unknown token problem.")
print("\nTokenization impact on LR (val accuracy):")
print(f"  Unigrams only:       0.8886")
print(f"  Unigrams + Bigrams:  0.8964  (+0.78%)")
print(f"\nPreprocessing impact on LR (val accuracy):")
print(f"  With stopwords:      0.8964")
print(f"  Without stopwords:   0.8970  (+0.06%, negligible)")
print("\nConclusion: Bigrams help capture phrase-level sentiment ('not good',")
print("'very bad'). Stopword removal has minimal effect because bigrams")
print("already encode negation context.")

TOKENIZATION STRATEGY COMPARISON SUMMARY
Strategy                       Vocab           OOV Handling         Used In
----------------------------------------------------------------------
TF-IDF Unigrams                50K sparse      Ignored              LogReg (baseline)
TF-IDF Unigrams+Bigrams        50K sparse      Ignored              LogReg (final)
Whitespace → integer IDs       30K             <UNK> token          BiLSTM
WordPiece subword              30K subword     Subword split        DistilBERT

Key observation: WordPiece handles OOV words by splitting into subwords
e.g. 'fanatic' → ['fan', '##atic'], eliminating unknown token problem.

Tokenization impact on LR (val accuracy):
  Unigrams only:       0.8886
  Unigrams + Bigrams:  0.8964  (+0.78%)

Preprocessing impact on LR (val accuracy):
  With stopwords:      0.8964
  Without stopwords:   0.8970  (+0.06%, negligible)

Conclusion: Bigrams help capture phrase-level sentiment ('not good',
'very bad'). Stopword removal has mi

In [16]:
# ---- Representation Types Comparison ----
print("=" * 70)
print("REPRESENTATION TYPES: PERFORMANCE & INTERPRETABILITY")
print("=" * 70)

print("""
1. SPARSE REPRESENTATIONS (TF-IDF + Logistic Regression)
   - Accuracy:        0.8913 (test)
   - Training time:   ~seconds
   - Interpretability: HIGH — top features directly inspectable
   - Limitation:      No word order, no semantic similarity
""")

# Show top positive/negative features for interpretability
import numpy as np
feature_names = tfidf.get_feature_names_out()
coefs = lr_model.coef_[0]
top_pos = np.argsort(coefs)[-10:][::-1]
top_neg = np.argsort(coefs)[:10]

print("   Top positive features:", [feature_names[i] for i in top_pos])
print("   Top negative features:", [feature_names[i] for i in top_neg])

print("""
2. DENSE SEQUENTIAL REPRESENTATIONS (BiLSTM)
   - Accuracy:        0.8622 (test)
   - Training time:   ~5 minutes (GPU)
   - Interpretability: LOW — hidden states not directly inspectable
   - Limitation:      Trained from scratch, no pretrained knowledge
                      Struggles with long-range dependencies

3. CONTEXTUAL REPRESENTATIONS (DistilBERT)
   - Accuracy:        0.9094 (test)
   - Training time:   ~25 minutes (GPU)
   - Interpretability: VERY LOW — 66M parameters, attention weights complex
   - Advantage:       Pretrained on large corpus, captures nuanced semantics
                      WordPiece handles OOV, attention handles long-range deps

SUMMARY TABLE:
""")

print(f"{'Model':<25} {'Test Acc':>10} {'Macro-F1':>10} {'Train Time':>12} {'Interpretability':>18}")
print("=" * 80)
print(f"{'TF-IDF + LogReg':<25} {test_acc_lr:>10.4f} {test_f1_lr:>10.4f} {'~seconds':>12} {'High':>18}")
print(f"{'BiLSTM':<25} {test_acc_bilstm:>10.4f} {test_f1_bilstm:>10.4f} {'~5 min':>12} {'Low':>18}")
print(f"{'DistilBERT':<25} {test_acc_bert:>10.4f} {test_f1_bert:>10.4f} {'~25 min':>12} {'Very Low':>18}")
print("=" * 80)
print("\nKey finding: Contextual embeddings (DistilBERT) outperform both sparse")
print("and sequential models, but at significant computational cost.")
print("Surprisingly, TF-IDF+LR outperforms BiLSTM — showing that strong")
print("sparse features can beat neural models trained from scratch.")

REPRESENTATION TYPES: PERFORMANCE & INTERPRETABILITY

1. SPARSE REPRESENTATIONS (TF-IDF + Logistic Regression)
   - Accuracy:        0.8913 (test)
   - Training time:   ~seconds
   - Interpretability: HIGH — top features directly inspectable
   - Limitation:      No word order, no semantic similarity

   Top positive features: ['great', 'excellent', 'wonderful', 'the best', 'best', 'perfect', 'love', 'amazing', 'loved', 'well']
   Top negative features: ['bad', 'worst', 'awful', 'the worst', 'boring', 'poor', 'waste', 'no', 'terrible', 'nothing']

2. DENSE SEQUENTIAL REPRESENTATIONS (BiLSTM)
   - Accuracy:        0.8622 (test)
   - Training time:   ~5 minutes (GPU)
   - Interpretability: LOW — hidden states not directly inspectable
   - Limitation:      Trained from scratch, no pretrained knowledge
                      Struggles with long-range dependencies

3. CONTEXTUAL REPRESENTATIONS (DistilBERT)
   - Accuracy:        0.9094 (test)
   - Training time:   ~25 minutes (GPU)
   - Inte